# Enterprise Policy Assistant — RAG Question Answering System

An end-to-end Retrieval-Augmented Generation (RAG) system for answering questions from the **Normandy Parish Council Employee Handbook**.

**Pipeline:** PDF → Text Cleaning → Chunking → Embeddings → Hybrid Retrieval → Cross-Encoder Reranking → FLAN-T5 Answer Generation

**Technologies:** Python, PyPDF, Sentence Transformers, ChromaDB, BM25, Cross-Encoder, FLAN-T5

## 1. Setup

In [ ]:
!pip install -q pypdf sentence-transformers chromadb rank_bm25 transformers sentencepiece langchain-text-splitters


In [ ]:
import re
import numpy as np
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import chromadb


## 2. Load the Employee Handbook

Upload the handbook in Colab. PDF page numbers are retained in chunk metadata so retrieved answers can be traced back to the source.

In [ ]:
from google.colab import files
uploaded = files.upload()
pdf_path = next(iter(uploaded))
reader = PdfReader(pdf_path)
print(f'Pages loaded: {len(reader.pages)}')


## 3. Clean and Normalize PDF Text

PDF extraction can split paragraphs across lines and introduce formatting noise. The cleaning stage reconstructs readable paragraphs while preserving bullets and important policy headings.

In [ ]:
import re

def clean_pdf_text(text):
    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Split into lines
    lines = text.split("\n")

    cleaned_lines = []

    for line in lines:
        line = line.strip()

        if not line:
            cleaned_lines.append("")
            continue

        # Normalize spaces
        line = re.sub(r"[ \t]+", " ", line)

        cleaned_lines.append(line)

    # Remove excessive blank lines
    cleaned_text = "\n".join(cleaned_lines)
    cleaned_text = re.sub(r"\n{3,}", "\n\n", cleaned_text)

    return cleaned_text.strip()


def join_wrapped_lines(text):
    # Split document into blocks using blank lines
    blocks = re.split(r"\n\s*\n", text)

    cleaned_blocks = []

    for block in blocks:
        lines = [line.strip() for line in block.split("\n") if line.strip()]

        if not lines:
            continue

        # Handle bullet points
        if any(line.startswith("•") for line in lines):
            bullet_lines = []
            current = ""

            for line in lines:

                if line.startswith("•"):
                    if current:
                        bullet_lines.append(current)

                    current = line

                else:
                    # Continuation of previous bullet
                    current += " " + line

            if current:
                bullet_lines.append(current)

            cleaned_blocks.extend(bullet_lines)

        else:
            # Join wrapped lines
            cleaned_blocks.append(" ".join(lines))

    return "\n\n".join(cleaned_blocks)


def restore_headings(text, headings):

    for heading in headings:

        text = re.sub(
            rf"\s*({re.escape(heading)})\s*",
            rf"\n\n\1\n\n",
            text,
            flags=re.IGNORECASE
        )

    # Remove excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def remove_page_markers(text):

    text = re.sub(
        r"Page\s*\|\s*\d+",
        "",
        text,
        flags=re.IGNORECASE
    )

    return re.sub(r"\n{3,}", "\n\n", text).strip()

In [ ]:
section_headings = [
    "Working from Home Policy",
    "Safe working environment",
    "Office equipment",
    "Hours of work"
]

## 4. Page-Aware Chunking

Each PDF page is cleaned and split into overlapping chunks.

- Chunk size: 800 characters
- Chunk overlap: 150 characters
- Contents page excluded
- Standalone divider headings excluded from retrieval

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

In [ ]:
def process_page(page_text, pdf_page_number, section_headings):

    # Step 1 — clean raw extraction
    cleaned = clean_pdf_text(page_text)

    # Step 2 — reconstruct paragraphs
    cleaned = join_wrapped_lines(cleaned)

    # Step 3 — restore known headings
    cleaned = restore_headings(
        cleaned,
        section_headings
    )

    # Step 4 — remove PDF page markers
    cleaned = remove_page_markers(cleaned)

    # Step 5 — remove excessive whitespace
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned).strip()

    # Step 6 — chunk
    chunks = text_splitter.split_text(cleaned)

    documents = []

    for i, chunk in enumerate(chunks):

        document_page = pdf_page_number - 1

        metadata = {
            "source": "employee_handbook.pdf",
            "pdf_page": pdf_page_number,
            "document_page": document_page,
            "chunk_id": f"page_{document_page}_chunk_{i+1}"
        }

        documents.append({
            "text": chunk,
            "metadata": metadata
        })

    return documents

In [ ]:
STANDALONE_HEADINGS = {"PART ONE POLICIES", "PART TWO FORMS AND TEMPLATES", "Sexual Harassment Policy"}

all_documents = []
for pdf_page_number, page in enumerate(reader.pages, start=1):
    if pdf_page_number == 2:
        continue
    raw_text = page.extract_text()
    if not raw_text:
        continue
    all_documents.extend(process_page(raw_text, pdf_page_number, section_headings))

documents = [doc for doc in all_documents if doc['text'].strip() not in STANDALONE_HEADINGS]
print(f'Total retrieval chunks: {len(documents)}')


## 5. Create Embeddings

In [ ]:
texts = [doc['text'] for doc in documents]
metadatas = [doc['metadata'] for doc in documents]
ids = [metadata['chunk_id'] for metadata in metadatas]

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedding_model.encode(texts, show_progress_bar=True)
print('Embedding shape:', embeddings.shape)


## 6. Store Chunks in ChromaDB

In [ ]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name='employee_handbook')
collection.add(ids=ids, documents=texts, embeddings=embeddings.tolist(), metadatas=metadatas)
print('Documents stored:', collection.count())


## 7. BM25 + Hybrid Retrieval

Semantic vector search captures meaning, while BM25 captures lexical overlap. The two signals are normalized and combined to create a hybrid ranking.

In [ ]:
from rank_bm25 import BM25Okapi
import re

In [ ]:
tokenized_corpus = [
    re.findall(r'\b\w+\b', text.lower())
    for text in texts
]

In [ ]:
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
def min_max_normalize(scores):

    min_score = min(scores)
    max_score = max(scores)

    if max_score == min_score:
        return [1.0] * len(scores)

    return [
        (score - min_score) / (max_score - min_score)
        for score in scores
    ]

In [ ]:
def distance_to_similarity(distances):
    normalized = min_max_normalize(distances)
    return [1 - score for score in normalized]


In [ ]:
def hybrid_retrieve(query, top_k=5, alpha=0.5):

    # ------------------------------------------------
    # 1. Semantic / Vector Search
    # ------------------------------------------------

    query_embedding = embedding_model.encode(
        [query]
    )[0]

    vector_results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    vector_ids = vector_results["ids"][0]
    vector_documents = vector_results["documents"][0]
    vector_metadatas = vector_results["metadatas"][0]
    vector_distances = vector_results["distances"][0]


    # ------------------------------------------------
    # 2. BM25 Search
    # ------------------------------------------------

    query_tokens = re.findall(
        r'\b\w+\b',
        query.lower()
    )

    bm25_scores = bm25.get_scores(query_tokens)

    bm25_top_indices = bm25_scores.argsort()[::-1][:top_k]


    # ------------------------------------------------
    # 3. Create a unified candidate set
    # ------------------------------------------------

    candidates = {}


    # Add vector results
    for i in range(len(vector_ids)):

        doc_id = vector_ids[i]

        candidates[doc_id] = {
            "document": vector_documents[i],
            "metadata": vector_metadatas[i],
            "vector_distance": vector_distances[i],
            "bm25_score": 0
        }


    # Add BM25 results
    for idx in bm25_top_indices:

        doc_id = ids[idx]

        if doc_id not in candidates:

            candidates[doc_id] = {
                "document": texts[idx],
                "metadata": metadatas[idx],
                "vector_distance": None,
                "bm25_score": bm25_scores[idx]
            }

        else:

            candidates[doc_id]["bm25_score"] = bm25_scores[idx]


    # ------------------------------------------------
    # 4. Normalize BM25 scores
    # ------------------------------------------------

    bm25_values = [
        item["bm25_score"]
        for item in candidates.values()
    ]

    normalized_bm25 = min_max_normalize(
        bm25_values
    )


    # ------------------------------------------------
    # 5. Normalize vector distances
    # ------------------------------------------------

    vector_values = [
        item["vector_distance"]
        for item in candidates.values()
        if item["vector_distance"] is not None
    ]

    if vector_values:

        vector_similarities = distance_to_similarity(
            vector_values
        )

    else:

        vector_similarities = []


    # Map vector similarity back to documents
    vector_counter = 0

    for item in candidates.values():

        if item["vector_distance"] is not None:

            item["vector_similarity"] = (
                vector_similarities[vector_counter]
            )

            vector_counter += 1

        else:

            item["vector_similarity"] = 0


    # ------------------------------------------------
    # 6. Add normalized BM25 score
    # ------------------------------------------------

    for i, item in enumerate(candidates.values()):

        item["bm25_similarity"] = normalized_bm25[i]


    # ------------------------------------------------
    # 7. Hybrid score
    # ------------------------------------------------

    for item in candidates.values():

        item["hybrid_score"] = (
            alpha * item["vector_similarity"]
            +
            (1 - alpha) * item["bm25_similarity"]
        )


    # ------------------------------------------------
    # 8. Sort
    # ------------------------------------------------

    ranked_results = sorted(
        candidates.values(),
        key=lambda x: x["hybrid_score"],
        reverse=True
    )


    return ranked_results[:top_k]

## 8. Cross-Encoder Reranking

The top hybrid candidates are reranked using a Cross-Encoder that scores the question and document together.

In [ ]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

In [ ]:
def rerank_results(query, results, top_k=3):

    # Create query-document pairs
    pairs = [
        [query, result["document"]]
        for result in results
    ]

    # Get Cross-Encoder scores
    scores = reranker.predict(pairs)

    # Add reranker score to each result
    for result, score in zip(results, scores):
        result["reranker_score"] = float(score)

    # Sort by reranker score
    reranked_results = sorted(
        results,
        key=lambda x: x["reranker_score"],
        reverse=True
    )

    # Return top K
    return reranked_results[:top_k]


## 9. Grounded Answer Generation

The model is instructed to answer only from retrieved handbook content and to return a fixed fallback when the answer is not supported by the context.

In [ ]:
def build_context(reranked_results):

    context_parts = []

    for i, result in enumerate(reranked_results, 1):

        page = result["metadata"]["pdf_page"]
        document = result["document"]

        context_parts.append(
            f"SOURCE {i} - Page {page}\n"
            f"{document}"
        )

    return "\n\n".join(context_parts)

In [ ]:
def build_prompt(query, context):
    prompt = f"""
You are answering questions about an employee handbook.

Use ONLY the information contained in the context.

Rules:
- Answer the exact question asked.
- Give the shortest complete answer that is supported by the context.
- Preserve important conditions or qualifications from the handbook.
- Do not add information that is not in the context.
- Do not combine unrelated rules.
- If the question asks "how many", give the number.
- If the question asks "how often", give the frequency.
- If the question asks "can", answer yes/no AND include an important qualification if one is explicitly stated.
- If the question asks what happens when a rule is violated,
  answer using a complete sentence from the relevant policy.
- Do not return an incomplete phrase such as "disciplinary action being taken".
- Do not provide a page number.
- Do not repeat the question.
- Do not provide explanations or commentary.
- If the answer cannot be found in the context, respond exactly:
I could not find the answer in the employee handbook.

Context:
{context}

Question:
{query}

Answer:
"""
    return prompt

In [ ]:
model_name = 'google/flan-t5-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
answer_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def generate_answer(prompt):
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048)
    outputs = answer_model.generate(**inputs, max_new_tokens=60, do_sample=False)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

def clean_answer(answer):
    answer = answer.strip()
    if answer.lower().startswith('answer:'):
        answer = answer[7:].strip()
    return answer


## 10. End-to-End RAG Pipeline

**Question → Hybrid Retrieval → Reranking → Context → Prompt → Answer**

In [ ]:
def run_pipeline(query, retrieval_top_k=5, rerank_top_k=3, alpha=0.5):
    results = hybrid_retrieve(query, top_k=retrieval_top_k, alpha=alpha)
    reranked = rerank_results(query, results, top_k=rerank_top_k)

    if not reranked:
        return {'answer': 'I could not find the answer in the employee handbook.', 'page': None, 'sources': []}

    context = build_context(reranked)
    prompt = build_prompt(query, context)
    answer = clean_answer(generate_answer(prompt))

    return {'answer': answer, 'page': reranked[0]['metadata']['pdf_page'], 'sources': reranked}

def answer_question(query):
    return run_pipeline(query)['answer']


## 11. Evaluation

The evaluation checks two things: **answer correctness** and **source-page correctness**. Page accuracy helps verify that the answer is grounded in the appropriate policy section.

In [ ]:
evaluation_data = [
    {
        "question": "How many days of annual leave can be carried over?",
        "expected_answer": "5 days",
        "expected_page": 20
    },
    {
        "question": "How often must the home working risk assessment be completed?",
        "expected_answer": "annually",
        "expected_page": 13
    },
    {
        "question": "How much daily rest must employees receive?",
        "expected_answer": "11 continuous hours",
        "expected_page": 13
    },
    {
        "question": "What happens if an employee violates the attendance policy?",
        "expected_answer": "disciplinary action",
        "expected_page": 22
    },
    {
        "question": "Can employees work from home?",
        "expected_answer": "yes",
        "expected_page": 13
    },
    {
        "question": "How many days of compassionate leave can an employee receive?",
        "expected_answer": "5 working days",
        "expected_page": 22
    }
]

def ask_handbook(query):
    return run_pipeline(query)

def evaluate_handbook(evaluation_data):
    results = []

    for item in evaluation_data:
        result = ask_handbook(item["question"])
        generated_answer = result["answer"].lower()
        expected_answer = item["expected_answer"].lower()

        results.append({
            "question": item["question"],
            "generated_answer": result["answer"],
            "expected_answer": item["expected_answer"],
            "generated_page": result["page"],
            "expected_page": item["expected_page"],
            "answer_correct": expected_answer in generated_answer,
            "page_correct": result["page"] == item["expected_page"]
        })

    return results

results = evaluate_handbook(evaluation_data)

answer_accuracy = sum(r["answer_correct"] for r in results) / len(results)
page_accuracy = sum(r["page_correct"] for r in results) / len(results)

print(f"Answer Accuracy: {answer_accuracy:.1%}")
print(f"Page Accuracy: {page_accuracy:.1%}")

## 12. Additional Unseen-Question Validation

A small set of differently worded questions is used as a qualitative sanity check after the formal evaluation.

In [ ]:
unseen_questions = [
    "What should an employee do if they are sick and cannot attend work?",
    "How much annual leave can normally be carried forward?",
    "What is the minimum break required during a working day over 6 hours?",
    "Can an employee use their own computer for home working?",
    "How many hours is the standard council working week?",
    "What is the maximum paid compassionate leave?"
]

for question in unseen_questions:
    result = ask_handbook(question)
    print(f"Q: {question}")
    print(f"A: {result['answer']}")
    print(f"Page: {result['page']}\n")

## 13. Final Interactive Q&A

The final interface accepts a question and returns the grounded answer. Type `exit` to stop.

In [ ]:
while True:
    question = input("Ask a question about the Employee Handbook (or type 'exit'): ")

    if question.strip().lower() == "exit":
        print("Goodbye!")
        break

    result = ask_handbook(question)
    print("\nAnswer:", result["answer"])
    print()

## Project Summary

This project demonstrates an end-to-end Retrieval-Augmented Generation (RAG) system:

**Document ingestion → PDF cleaning → page-aware chunking → sentence embeddings → ChromaDB vector search → BM25 lexical search → hybrid retrieval → Cross-Encoder reranking → grounded FLAN-T5 answer generation → evaluation**

The system is designed to answer questions only from the supplied employee handbook and retain source-page metadata for traceability.